# 02 — Training: 24h-ahead PECO demand

Walk-forward + hyperparameter search + LightGBM/XGBoost ensemble.
No random split. No lag_1 (that would leak for a 24h horizon).

**File:** `data/processed/peco_features.csv`  
Upload that CSV next to this notebook (same as EDA). Send back the metrics table.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from xgboost import XGBRegressor

CSV_PATH = Path("../data/processed/peco_features.csv")
if not CSV_PATH.exists():
    CSV_PATH = Path("data/processed/peco_features.csv")
if not CSV_PATH.exists():
    CSV_PATH = Path("peco_features.csv")

FEATURE_COLS = [
    "hour_sin", "hour_cos", "dow_sin", "dow_cos", "month_sin", "month_cos",
    "is_weekend", "is_holiday",
    "temperature_2m", "relative_humidity_2m", "wind_speed_10m",
    "precipitation", "cloud_cover", "shortwave_radiation",
    "hdd", "cdd", "temp_change_24",
    "lag_24", "lag_48", "lag_72", "lag_168",
    "roll_mean_24", "roll_std_24", "roll_min_24", "roll_max_24", "roll_mean_168",
    "was_imputed",
]
TARGET = "demand_mw"

df = pd.read_csv(CSV_PATH)
df["ts_utc"] = pd.to_datetime(df["ts_utc"], utc=True)
df = df.sort_values("ts_utc").reset_index(drop=True)
print("rows", df.shape, "file", CSV_PATH.resolve())
print(df["split"].value_counts())

rows (40662, 31) file /content/peco_features.csv
split
train    23161
val       8760
test      8741
Name: count, dtype: int64


In [3]:
def mae(y, p):
    return float(np.mean(np.abs(np.asarray(y) - np.asarray(p))))

def rmse(y, p):
    return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(p)) ** 2)))

def wape(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return float(np.sum(np.abs(y - p)) / np.sum(np.abs(y)))

def peak_mae(y, p, thr):
    y, p = np.asarray(y, float), np.asarray(p, float)
    m = y >= thr
    return mae(y[m], p[m]) if m.any() else float("nan")

def scores(y, p, thr):
    return {"wape": wape(y, p), "mae": mae(y, p), "rmse": rmse(y, p), "peak_mae": peak_mae(y, p, thr)}

train = df[df["split"] == "train"]
val = df[df["split"] == "val"]
test = df[df["split"] == "test"]
X_train, y_train = train[FEATURE_COLS], train[TARGET]
X_val, y_val = val[FEATURE_COLS], val[TARGET]
X_test, y_test = test[FEATURE_COLS], test[TARGET]
P90 = float(y_train.quantile(0.90))
print("train P90 MW (peak gate):", P90)

train P90 MW (peak gate): 5710.0


## Baselines (seasonal naive)

In [4]:
rows = []
for name, col in [("seasonal_naive_24", "lag_24"), ("seasonal_naive_168", "lag_168")]:
    for split_name, part in [("val", val), ("test", test)]:
        s = scores(part[TARGET], part[col], P90)
        s.update(model=name, split=split_name)
        rows.append(s)
pd.DataFrame(rows)

,wape,mae,rmse,peak_mae,model,split
0,0.069234,305.601027,435.097353,548.992788,seasonal_naive_24,val
1,0.070727,317.607825,436.406694,506.122407,seasonal_naive_24,test
2,0.115183,508.423174,732.002588,1035.747596,seasonal_naive_168,val
3,0.111709,501.642489,712.153290,989.991701,seasonal_naive_168,test


## LightGBM + XGBoost (walk-forward RandomizedSearch)

In [5]:
tscv = TimeSeriesSplit(n_splits=3)

lgb = LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1)
lgb_search = RandomizedSearchCV(
    lgb,
    param_distributions={
        "n_estimators": [200, 400, 600],
        "learning_rate": [0.03, 0.05, 0.1],
        "num_leaves": [31, 63, 127],
        "min_child_samples": [20, 40, 80],
        "subsample": [0.7, 0.9, 1.0],
    },
    n_iter=8,
    cv=tscv,
    scoring="neg_mean_absolute_error",
    random_state=42,
    n_jobs=-1,
)
lgb_search.fit(X_train, y_train)
print("LGBM", lgb_search.best_params_, "cv MAE", -lgb_search.best_score_)

xgb = XGBRegressor(random_state=42, n_jobs=-1, tree_method="hist")
xgb_search = RandomizedSearchCV(
    xgb,
    param_distributions={
        "n_estimators": [200, 400, 600],
        "learning_rate": [0.03, 0.05, 0.1],
        "max_depth": [4, 6, 8],
        "min_child_weight": [5, 10, 20],
        "subsample": [0.7, 0.9, 1.0],
    },
    n_iter=8,
    cv=tscv,
    scoring="neg_mean_absolute_error",
    random_state=42,
    n_jobs=-1,
)
xgb_search.fit(X_train, y_train)
print("XGB", xgb_search.best_params_, "cv MAE", -xgb_search.best_score_)

LGBM {'subsample': 1.0, 'num_leaves': 127, 'n_estimators': 400, 'min_child_samples': 80, 'learning_rate': 0.05} cv MAE 129.16368338070345
XGB {'subsample': 0.7, 'n_estimators': 600, 'min_child_weight': 5, 'max_depth': 4, 'learning_rate': 0.03} cv MAE 126.09975615453637


## Ensemble (mean of tuned LGBM + XGB) and comparison

In [6]:
lgb_model = lgb_search.best_estimator_
xgb_model = xgb_search.best_estimator_

def add_model_rows(name, pred_val, pred_test):
    out = []
    for split_name, y, p in [("val", y_val, pred_val), ("test", y_test, pred_test)]:
        s = scores(y, p, P90)
        s.update(model=name, split=split_name)
        out.append(s)
    return out

rows.extend(add_model_rows("lightgbm", lgb_model.predict(X_val), lgb_model.predict(X_test)))
rows.extend(add_model_rows("xgboost", xgb_model.predict(X_val), xgb_model.predict(X_test)))
ens_val = 0.5 * lgb_model.predict(X_val) + 0.5 * xgb_model.predict(X_val)
ens_test = 0.5 * lgb_model.predict(X_test) + 0.5 * xgb_model.predict(X_test)
rows.extend(add_model_rows("ensemble_mean", ens_val, ens_test))

metrics = pd.DataFrame(rows).sort_values(["split", "wape"])
display(metrics)
print("\nPrimary metric is WAPE. Also watch Peak-MAE. Send this table back.")

,wape,mae,rmse,peak_mae,model,split
9,0.028887,129.721321,185.936918,287.251132,ensemble_mean,test
5,0.029758,133.631939,191.432075,289.696816,lightgbm,test
7,0.030117,135.241775,190.537883,290.585416,xgboost,test
1,0.070727,317.607825,436.406694,506.122407,seasonal_naive_24,test
3,0.111709,501.642489,712.153290,989.991701,seasonal_naive_168,test
8,0.025907,114.354115,161.593119,224.687473,ensemble_mean,val
4,0.026987,119.121676,169.503376,230.344798,lightgbm,val
6,0.027266,120.354372,165.750215,228.117984,xgboost,val
0,0.069234,305.601027,435.097353,548.992788,seasonal_naive_24,val
2,0.115183,508.423174,732.002588,1035.747596,seasonal_naive_168,val



Primary metric is WAPE. Also watch Peak-MAE. Send this table back.
